In [13]:

using Revise
using PVlib

using HTTP
using JSON
using DataFrames
using Dates
using CSV
using Plots
using Statistics
using LinearAlgebra
using TimeZones

using Optim
using Zygote

In [14]:
# testing panel tilt azimuth with Zygote

function panel_tilt_azimuth_zygote(
    motion::AbstractMatrix{<:Real},
    install_tilt_deg::Real = 0.0,
    install_azimuth_deg::Real = 0.0,
    yaw_offset_deg::Real = 0.0,
)
    roll = motion[:, 4]
    pitch = motion[:, 5]
    yaw = motion[:, 6] .+ deg2rad(yaw_offset_deg)

    β = deg2rad(install_tilt_deg)
    γ = deg2rad(install_azimuth_deg)

    n0x = sin(β) * sin(γ)
    n0y = sin(β) * cos(γ)
    n0z = cos(β)

    cy = cos.(yaw)
    sy = sin.(yaw)
    cp = cos.(pitch)
    sp = sin.(pitch)
    cr = cos.(roll)
    sr = sin.(roll)

    nx = (cy .* cp) .* n0x .+
         (cy .* sp .* sr .- sy .* cr) .* n0y .+
         (cy .* sp .* cr .+ sy .* sr) .* n0z

    ny = (sy .* cp) .* n0x .+
         (sy .* sp .* sr .+ cy .* cr) .* n0y .+
         (sy .* sp .* cr .- cy .* sr) .* n0z

    nz = (-sp) .* n0x .+
         (cp .* sr) .* n0y .+
         (cp .* cr) .* n0z

    nh = sqrt.(nx.^2 .+ ny.^2)
    tilt = atan.(nh, PVlib.pv_smooth_abs.(nz))
    azimuth = atan.(nx, ny)   # unwrapped

    return rad2deg.(tilt), rad2deg.(azimuth), nz
end

panel_tilt_azimuth_zygote (generic function with 4 methods)

In [15]:
loss(install_tilt_deg, install_azimuth_deg, yaw_offset_deg, motion) = begin
    tilt, azimuth, nz = panel_tilt_azimuth_zygote(
        motion,
        install_tilt_deg,
        install_azimuth_deg,
        yaw_offset_deg,
    )
    sum(tilt) + sum(azimuth) + sum(nz)
end

motion = rand(10, 6)

grads = gradient(
    (it, ia, yo) -> loss(it, ia, yo, motion),
    20.0,   # install_tilt_deg
    180.0,  # install_azimuth_deg
    5.0,    # yaw_offset_deg
)

println(grads)

(15.297837336895908, -0.12113448550374742, -10.000000000000002)


In [16]:
latitude = 35.1
longitude = -106.6
altitude = 1500.0 # m

api_key = "E52b7mqeTWLigj2xF5Bn4n6Mm87ecm5LFFeYh4US" # from NLR
email = "jtgrasb@sandia.gov"

start_monthday=(1, 1)
end_monthday=(1, 1)

number_of_panels = 1
surface_azimuth = 180.0
surface_tilt = latitude

# try zygote one function at a time
weather_data = get_meteorological_data_nsrdb(latitude, longitude, api_key, email, start_monthday, end_monthday, false)
pv_module   = read_solar_module("Canadian Solar CS5P-220M [ 2009]")
pv_inverter = read_solar_inverter("ABB: MICRO-0.25-I-OUTD-US-208 [208V]")

function ac_power_return(latitude, longitude, altitude, surface_tilt, surface_azimuth, weather_data, pv_module, pv_inverter)

    sol_position = get_solar_position(latitude, longitude, altitude, weather_data)
    total_irradiance = get_total_irradiance(surface_tilt, surface_azimuth, weather_data, sol_position, 0.25, weather_data.time) 
    cell_temp = sapm_cell_temperature(total_irradiance, weather_data)
    #cell_temp = rolling_average_sapm_cell_temperature(total_irradiance, weather_data)

    effective_irradiance = sapm_effective_irradiance(total_irradiance, pv_module, sol_position, surface_tilt, surface_azimuth, altitude)

    panel = Panel(0.526, 0.652) # panel dimensnions in meters (Kyocera Solar KC40T [2008 (E)])
    # A box above the panel
    obstacle = BoxObstacle(
        -0.5, 0.5,   # x-range
        -0.3, 0.3,     # y-range
        0.4, 0.5   # z-range
    )
    n_cells_per_column = 9
    f_shade = get_shaded_fraction(sol_position, surface_tilt, surface_azimuth, panel, obstacle, nx=10, ny=10)
    power_norm = get_power_norm(total_irradiance, f_shade, n_cells_per_column)

    dc_components = sapm_dc_components(pv_module, effective_irradiance, cell_temp)
    ac_power = sandia_ac_power(pv_inverter, dc_components)

    return getfield(ac_power, :ac_power)
end

ac_power_res = ac_power_return(latitude, longitude, altitude, surface_tilt, surface_azimuth, weather_data[12], pv_module, pv_inverter)

mean_power_z = orientation -> -mean(map(w -> ac_power_return(latitude, longitude, altitude, surface_tilt, orientation[1], w, pv_module, pv_inverter), weather_data))

g = Zygote.gradient(mean_power_z, [surface_azimuth])[1]
println(g)

[0.15865211884656416]


In [17]:
function g!(G, orientation)
    G[:] = Zygote.gradient(mean_power_z, orientation)[1]
end

x0 = [180.0]
res = optimize(mean_power_z, g!, x0, BFGS())

# Display the results
println("Optimal x: ", res.minimizer[1])  # Optimal value of x
println("Maximum value of f: ", -res.minimum)  # Minimum value of f

Optimal x: 162.4588154882986
Maximum value of f: 51.857716733386575


In [18]:
latitude = 35.1
longitude = -106.6
altitude = 1500.0 # m

date = Date(2023, 1, 1)
tz = TimeZone("America/Denver")

number_of_panels = 1
surface_azimuth = 180
surface_tilt = latitude

function objective(latitude, longitude, altitude, surface_tilt, surface_azimuth, weather_data, pv_module, pv_inverter)

    avg_power = mean(map(w -> ac_power_return(latitude, longitude, altitude, surface_tilt, surface_azimuth, w, pv_module, pv_inverter), weather_data))
    return avg_power
end

initial_azimuth = [180.0]
objective_result(vars) = -objective(latitude, longitude, altitude, surface_tilt, vars[1], weather_data, pv_module, pv_inverter)
result = optimize(objective_result, initial_azimuth)

# Display the results
println("Optimal x: ", result.minimizer[1])  # Optimal value of x
println("Maximum value of f: ", -result.minimum)  # Minimum value of f

Optimal x: 162.4595760345459
Maximum value of f: 51.85771673077206


In [19]:
time = range(0, 500, 1001)
amplitude = [1, 0.1, 1, 0.1, 1, 0.2]
period = 10 # seconds
ω = 2π / period
WEC_response = amplitude' .* sin.(ω .* time)

initial_tilt = 20.0
initial_azimuth = 180.0

tilt1, az1, nz1 = panel_tilt_azimuth(WEC_response, initial_tilt, initial_azimuth)
tilt2, az2, nz2 = panel_tilt_azimuth_zygote(WEC_response, initial_tilt, initial_azimuth)

println("max |tilt diff| = ", maximum(abs.(tilt1 .- tilt2)))
println("max |nz diff|   = ", maximum(abs.(nz1 .- nz2)))

max |tilt diff| = 0.0
max |nz diff|   = 0.0


In [20]:
println("max |sin diff| = ", maximum(abs.(sin.(deg2rad.(az1)) .- sin.(deg2rad.(az2)))))
println("max |cos diff| = ", maximum(abs.(cos.(deg2rad.(az1)) .- cos.(deg2rad.(az2)))))

max |sin diff| = 3.3306690738754696e-16
max |cos diff| = 1.0824674490095276e-15


In [21]:
sol_position = get_solar_position(latitude, longitude, altitude, weather_data)
albedo = 0.2

total1 = get_total_irradiance(tilt1, az1, weather_data[12], sol_position[12], albedo, time)
total2 = get_total_irradiance(tilt2, az2, weather_data[12], sol_position[12], albedo, time)

for f in fieldnames(typeof(total1[1]))
    a = getfield.(total1, f)
    b = getfield.(total2, f)
    println("max diff for ", f, " = ", maximum(abs.(a .- b)))
end

max diff for time = 0 milliseconds
max diff for poa_global = 7.389644451905042e-13
max diff for poa_direct = 6.821210263296962e-13
max diff for poa_diffuse = 4.263256414560601e-14
max diff for poa_sky_diffuse = 4.263256414560601e-14
max diff for poa_ground_diffuse = 0.0
